In [0]:
kafka_bootstrap = 'Kafka Server'
kafka_api_key = 'Kafka API Key'
kafka_api_secret = 'Kafka Secret Key'

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp, lit
from pyspark.sql.types import *

In [0]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap) \
    .option("subscribe", "smartgear_orders") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", f"""
        kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required
        username='{kafka_api_key}'
        password='{kafka_api_secret}';
    """) \
    .option("startingOffsets", "earliest") \
    .load()

In [0]:
df_raw = df_kafka.selectExpr("CAST(value AS STRING) as json_value")

In [0]:
schema = StructType([
    StructField("order_id", StringType()),
    StructField("order_number", LongType()),
    StructField("timestamp", StringType()),
    StructField("store_id", StringType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price", DoubleType()),
    StructField("region", StringType())
])

In [0]:
df_parsed = df_raw.withColumn("data", from_json(col("json_value"), schema)) \
                  .select("data.*")

In [0]:
df_bronze = df_parsed \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source", lit("kafka"))

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS smartgear;
CREATE VOLUME IF NOT EXISTS workspace.smartgear.checkpoints;

In [0]:
df_bronze.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/smartgear/checkpoints/bronze_orders") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("workspace.smartgear.bronze_orders")

In [0]:
%sql
SELECT COUNT(*) FROM workspace.smartgear.bronze_orders;

COUNT(*)
31


True